# V16: frozen preference rewards and later human votes

**FINAL: watchdog stopped the service (ERROR); full candidate quality is incomplete.**
Retain official0.82987/rank23; the rank<=19objective remains active.

Replace one expert with a pretrained scalar reward head and fit only a symmetric
three-parameter A/B/tie mapping. Native conversations preserve turns; each answer
is evaluated independently. Main8B/control0.6B, fixed protocol and weights.
The fresh1024groups postdate both published reward checkpoints and exclude
previously inspected groups. This is chronological source evidence, not Kaggle CV.
Text can preexist its new vote, and newer users/models may not represent hidden
competition data. See [the protocol](../docs/SPAN_016_PILOT_PROTOCOL.md).

In [1]:
from pathlib import Path
import json
import pandas as pd
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PUB=REPO/'results/span_016'
def read(name):return json.loads((PUB/name).read_text())
q=read('source_observation.json');audit=read('source_audit.json')
display(pd.DataFrame([{'size':m['size'],'repository':m['repository'],'revision':m['revision'],
 'weight_bytes':m['total_weight_bytes'],'license':m['license'],'gated':m['gated'],
 'scalar_outputs':len(m['config']['id2label'])} for m in q['models']]))
print('Source stage passed:',audit['source_stage_passed'])
print('Source-stage GPU admission:',audit['gpu_admitted'])

,size,repository,revision,weight_bytes,license,gated,scalar_outputs
0,8B,Skywork/Skywork-Reward-V2-Qwen3-8B,6f19fdefb933293d4898bdb59a96f7223d998659,15136865264,apache-2.0,False,1
1,0.6B,Skywork/Skywork-Reward-V2-Qwen3-0.6B,8c14a4e9e6321deaf572544339b16b8d6bbe8886,1192137232,apache-2.0,False,1


Source stage passed: True
Source-stage GPU admission: False


In [2]:
cohort=read('matched_cohort_manifest.json') if (PUB/'matched_cohort_manifest.json').exists() else read('cohort_manifest.json')
display(pd.DataFrame([{k:cohort.get(k) for k in ['rows','available_groups','excluded_groups',
 'post_publication_cutoff','timestamp_range','selected_bin_counts']}]))
assert cohort['targets_read_after_durable_selection'] and not cohort.get('uses_targets_for_selection',cohort.get('selection_uses_targets',False))
if (PUB/'matched_token_reference.json').exists():
 initial=read('token_reference.json');matched=read('matched_token_reference.json')
 display(pd.DataFrame([{'population':'initial unused source','tokens_per_pair':initial['cohorts']['long']['mean_pair_tokens'],'work_ratio':initial['source_to_target_reward_work_ratio']},
 {'population':'actual matched source','tokens_per_pair':matched['mean_pair_tokens'],'work_ratio':matched['source_to_target_reward_work_ratio']},
 {'population':'competition runtime512','tokens_per_pair':initial['cohorts']['uniform']['mean_pair_tokens'],'work_ratio':1.}]))
native=read('pinned_runtime_cpu_validation.json')
display(pd.DataFrame([{k:native[k] for k in ['passed','transformers','tokenizers','huggingface_hub','safetensors']}]))
print('Native rendering matches the card; prefix truncation retains final EOS/newline.')
print('Swap symmetry, gradients and held-out-group isolation were tested on CPU.')
if (PUB/'class_prior_audit.json').exists():
 prior=read('class_prior_audit.json')
 display(pd.DataFrame({'class':prior['classes'],
  'competition_fraction':prior['original_class_priors'],
  'later_vote_fraction':prior['later_vote_class_priors']}))
 print('Class-frequency diagnostic only; the frozen recipe and quality gates are unchanged.')

,rows,available_groups,excluded_groups,post_publication_cutoff,timestamp_range,selected_bin_counts
0,1024,9706,3072,2025-07-06T18:16:33+00:00,"[2025-07-06T18:51:18.949840+00:00, 2025-07-24T...","[244, 248, 219, 207, 55, 33, 11, 7]"


,population,tokens_per_pair,work_ratio
0,initial unused source,1785.453125,2.360143
1,actual matched source,739.095703,0.976991
2,competition runtime512,756.501953,1.000000


,passed,transformers,tokenizers,huggingface_hub,safetensors
0,True,4.52.3,0.21.1,0.31.4,0.5.3


Native rendering matches the card; prefix truncation retains final EOS/newline.
Swap symmetry, gradients and held-out-group isolation were tested on CPU.


,class,competition_fraction,later_vote_fraction
0,winner_model_a,0.349079,0.370117
1,winner_model_b,0.341911,0.337891
2,winner_tie,0.309011,0.291992


Class-frequency diagnostic only; the frozen recipe and quality gates are unchanged.


In [3]:
rehearsal=read('reward_pilot_cell_rehearsals.json')
print('Actual serialized worker rehearsals passed:',rehearsal['passed'])
print('Actual pipeline CPU bookkeeping tests:',rehearsal['actual_pipeline_tests'])
for name in ['asset_preparation.json','asset_upload.json','matched_input_upload.json',
 'jobs/reward_pilot/status.json','reward_pilot_source_verification.json']:
 if (PUB/name).exists():
  print(name);display(pd.DataFrame([read(name)]))
 else:print(name,': pending; no corresponding result is claimed')
if (PUB/'H56_information.json').exists():
 info=read('H56_information.json')
 display(pd.DataFrame([{'model':name,'log_loss':m['log_loss'],'accuracy':m['accuracy'],
  **m['class_log_loss']} for name,m in info['models'].items()]))
 display(pd.DataFrame([{k:info[k] for k in ['loss_reduction','paired_group95',
  'information_gate_passed','stronger_quality_gate_passed','official_submission_eligible','decision']}]))
elif (PUB/'H56_partial_diagnostic.json').exists():
 partial=read('H56_partial_diagnostic.json')
 assert partial['service_status']=='ERROR' and not partial['primary_comparison_complete']
 assert not partial['official_submission_eligible']
 print('FINAL: failed service preserved; complete expert diagnostics only. No submission.')
 display(pd.DataFrame([{'model':name,'log_loss':m['log_loss'],'accuracy':m['accuracy'],
  **m['class_log_loss']} for name,m in partial['models'].items()]))
 display(pd.DataFrame([partial['common_off_route']]))
 print('The common off-route subset is exploratory; the full paired loss is unmeasured.')
 print('Possible completion-gain bounds:',partial['frozen_recipe_completion_gain_bounds'])
 print('Those bounds are diagnostics and cannot qualify a candidate.')
 display(pd.DataFrame([{'size':size,'cold_seconds':s['cold_seconds'],
  'warmup_seconds':s['cohorts']['warmup']['seconds'],
  'source1024_seconds':s['cohorts']['long']['seconds'],
  'peak_GiB':[round(x/2**30,3) for x in s['cohorts']['long']['peak_allocated_bytes']]}
  for size,s in partial['native_inference'].items()]))
 print('Charged GPU seconds:',partial['charged_gpu_seconds'])
 print('Owned worker:',partial['session'])
 print('Artifacts verified:',partial['artifact_hashes_verified'])
 print('Retain official0.82987. Full quality/runtime gates were not met.')
else:print('H56_information.json: pending; actual model quality has not been audited.')
print('A source/CPU check is not a GPU quality estimate or a leaderboard improvement.')

Actual serialized worker rehearsals passed: True
Actual pipeline CPU bookkeeping tests: 5
asset_preparation.json


,at,prepared,dataset,private_requested,total_bytes,files_sha256,actual_weight_bytes,all_expected_weight_hashes_verified,known_external_training_votes_included,hidden_votes_included,new_gpu_executions
0,2026-11-08T04:30:22.366913+00:00,True,arseniytrushin/llm-span016-reward-assets/1,True,16380017198,{'rm8b__tokenizer_config.json': 'd3e08cbc53421...,16329002496,True,True,False,0


asset_upload.json


,at,dataset,ready,status,private_verified,all_weight_hashes_verified,asset_manifest_sha256,new_gpu_executions
0,2026-11-08T05:37:09.462492+00:00,arseniytrushin/llm-span016-reward-assets/1,True,ready,True,True,401b05987eab244bc311a2d5031c30558a7d21becb52b8...,0


matched_input_upload.json


,at,dataset,ready,status,private_verified,input_manifest_sha256,new_gpu_executions
0,2026-11-08T05:38:09.773589+00:00,arseniytrushin/llm-span016-lengthmatched-inputs/1,True,ready,True,f29b9fe35f2e9e1d331913067fb540db74e248230c8fd1...,0


... (6 outputs left out)

,rows,loss_reduction,paired_group95,full_cohort,post_hoc_subset,no_predictions_imputed
0,716,0.002101,"[-0.009165, 0.012619]",False,True,True


The common off-route subset is exploratory; the full paired loss is unmeasured.
Possible completion-gain bounds: [-0.08559347657625413, 0.07537215560123582]
Those bounds are diagnostics and cannot qualify a candidate.


,size,cold_seconds,warmup_seconds,source1024_seconds,peak_GiB
0,8B,24.583803,45.424709,734.457467,"[8.419, 7.228]"
1,0.6B,4.794881,7.505914,100.349378,[1.503]


Charged GPU seconds: 3492.4609999999993
Owned worker: {'seconds': 3480.0260894298553, 'exit_code': -15, 'role': 'reward_pilot'}
Artifacts verified: 31
Retain official0.82987. Full quality/runtime gates were not met.
A source/CPU check is not a GPU quality estimate or a leaderboard improvement.
